---

<h1 style="text-align: center;"><strong>Data Science & Statistical Computing</strong></h1>
<h2 style="text-align: center;"><strong>Aula 03 - Probabilidade e Tópicos de Amostragem</strong></h2>
<h3 style="text-align: center;"><strong>Prof. Jones Egydio</strong></h3>
<h4 style="text-align: center;"><strong>FIAP - 2026</strong></h4>
<br>
<br>


---

# **Aula 3: Probabilidade e Tópicos de Amostragem**

## **Objetivos**
- Compreender os conceitos fundamentais de probabilidade aplicados à ciência de dados.
- Identificar espaço amostral, eventos e relações entre eventos em problemas práticos.
- Aplicar probabilidade condicional, regra do produto, regra da soma e teorema de Bayes.
- Compreender métodos de amostragem, representatividade e viés de amostragem.
- Implementar exemplos em Python com um dataset público e interpretar os resultados.

## **Dataset**
Vamos usar novamente o dataset iris, que contém medidas de flores de três espécies: *setosa*, *versicolor* e *virginica*. Esse conjunto é adequado para a aula porque permite construir eventos probabilísticos, comparar grupos e simular diferentes estratégias de amostragem de forma clara e reproduzível.


In [1]:
# imports e carregamento do dataset iris
import math
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame.copy()

# renomeando para ficar mais didático
df = df.rename(columns={
    "sepal length (cm)": "sepal_length_cm",
    "sepal width (cm)": "sepal_width_cm",
    "petal length (cm)": "petal_length_cm",
    "petal width (cm)": "petal_width_cm",
    "target": "species_id"
})

# mapeando id -> nome da espécie
df["species"] = df["species_id"].map(dict(enumerate(iris.target_names)))

# evento auxiliar para vários exemplos da aula
df["petal_length_gt_5"] = df["petal_length_cm"] > 5.0

df.head()

,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species_id,species,petal_length_gt_5
0,5.1,3.5,1.4,0.2,0,setosa,False
1,4.9,3.0,1.4,0.2,0,setosa,False
2,4.7,3.2,1.3,0.2,0,setosa,False
3,4.6,3.1,1.5,0.2,0,setosa,False
4,5.0,3.6,1.4,0.2,0,setosa,False


---

## **1 Probabilidade**
Probabilidade é a linguagem matemática usada para quantificar incerteza. Em ciência de dados, ela aparece em classificação, inferência estatística, avaliação de risco, testes diagnósticos, previsão e tomada de decisão sob incerteza.

Nesta aula, trabalharemos a probabilidade como proporção de casos favoráveis dentro de um conjunto de possibilidades bem definidas. Em vários exemplos, vamos interpretar a probabilidade de selecionar observações do dataset iris com certas características.


## **1.1 Espaço amostral**
O espaço amostral é o conjunto de todos os resultados possíveis de um experimento aleatório. Ele é normalmente representado por $\Omega$.

**Fórmula:**
$$
P(A) = \frac{|A|}{|\Omega|}
$$

**Significado dos termos:**
- $\Omega$: conjunto de todos os resultados possíveis;
- $A$: evento de interesse;
- $|A|$: número de resultados favoráveis ao evento $A$;
- $|\Omega|$: número total de resultados possíveis.

No contexto desta aula, se escolhermos uma linha do dataset ao acaso, o espaço amostral será formado pelas 150 observações do iris.


In [2]:
# exemplo: espaço amostral ao selecionar uma observação do iris ao acaso
n_observacoes = len(df)
especies = df["species"].unique().tolist()

print(f"Tamanho do espaço amostral: {n_observacoes}")
print(f"Espécies presentes: {especies}")


Tamanho do espaço amostral: 150
Espécies presentes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


### **1.1.2 Eventos**
Um evento é qualquer subconjunto do espaço amostral. Em problemas aplicados, um evento descreve uma condição de interesse, como selecionar uma flor da espécie *setosa* ou uma observação com comprimento de pétala maior que 5 cm.

**Fórmula:**
$$
A \subseteq \Omega
$$

Se o experimento for equiprovável, então:
$$
P(A) = \frac{|A|}{|\Omega|}
$$

Eventos podem ser simples, quando descrevem uma única condição, ou compostos, quando combinam múltiplas condições.


In [3]:
# exemplo: evento A = "a flor pertence à espécie setosa"
evento_A = df["species"] == "setosa"
prob_A = evento_A.mean()

print(f"Quantidade de casos favoráveis: {evento_A.sum()}")
print(f"Probabilidade empírica de selecionar uma setosa: {prob_A:.3f}")


Quantidade de casos favoráveis: 50
Probabilidade empírica de selecionar uma setosa: 0.333


### **1.1.3 Eventos dependentes e independentes**
Dois eventos são independentes quando a ocorrência de um não altera a probabilidade do outro. Caso contrário, são dependentes.

**Fórmula da independência:**
$$
P(A \cap B) = P(A)P(B)
$$

**Significado dos termos:**
- $P(A \cap B)$: probabilidade de ocorrência simultânea de $A$ e $B$;
- $P(A)$: probabilidade de $A$;
- $P(B)$: probabilidade de $B$.

Se a igualdade não se verifica, os eventos não são independentes. Em dados reais, essa verificação costuma ser aproximada e interpretativa.


In [4]:
# exemplo: A = "species = setosa" e B = "petal_length_cm > 5"
A = df["species"] == "setosa"
B = df["petal_length_cm"] > 5

p_A = A.mean()
p_B = B.mean()
p_AeB = (A & B).mean()
produto = p_A * p_B

resultado = pd.Series({
    "P(A)": p_A,
    "P(B)": p_B,
    "P(A ∩ B)": p_AeB,
    "P(A)P(B)": produto
})
resultado


P(A)        0.333333
P(B)        0.280000
P(A ∩ B)    0.000000
P(A)P(B)    0.093333
dtype: float64

### **1.1.4 Arranjo e combinação**
Arranjos e combinações são ferramentas de contagem. Elas são fundamentais porque muitas probabilidades podem ser obtidas pela razão entre o número de casos favoráveis e o número total de casos possíveis.

**Arranjo simples:** número de formas de escolher e ordenar $p$ elementos dentre $n$ elementos distintos.
$$
A_{n,p} = \frac{n!}{(n-p)!}
$$

**Combinação simples:** número de formas de escolher $p$ elementos dentre $n$ elementos distintos sem considerar a ordem.
$$
C_{n,p} = \binom{n}{p} = \frac{n!}{p!(n-p)!}
$$

**Significado dos termos:**
- $n$: número total de elementos disponíveis;
- $p$: número de elementos escolhidos;
- $!$: fatorial.

Em ciência de dados, essas ideias aparecem em seleção de atributos, análise combinatória de experimentos e cálculo de probabilidades discretas.


In [5]:
# exemplo: de quantas formas podemos escolher 2 espécies dentre as 3 do iris?
n = 3
p = 2

arranjo = math.factorial(n) // math.factorial(n - p)
combinacao = math.comb(n, p)

print(f"Arranjo A(3,2): {arranjo}")
print(f"Combinação C(3,2): {combinacao}")
print("Combinações possíveis:", list(itertools.combinations(especies, 2)))


Arranjo A(3,2): 6
Combinação C(3,2): 3
Combinações possíveis: [(np.str_('setosa'), np.str_('versicolor')), (np.str_('setosa'), np.str_('virginica')), (np.str_('versicolor'), np.str_('virginica'))]


### **1.1.5 Eventos compostos**
Eventos compostos resultam da combinação de dois ou mais eventos simples. As operações mais comuns são interseção, união e complementar.

**Fórmulas principais:**
$$
A \cap B
$$
$$
A \cup B
$$
$$
A^c
$$

**Significado dos termos:**
- $A \cap B$: ocorrência simultânea de $A$ e $B$;
- $A \cup B$: ocorrência de pelo menos um dos eventos;
- $A^c$: não ocorrência de $A$.

Essas operações estruturam grande parte do raciocínio probabilístico em problemas aplicados.


In [6]:
# exemplo: evento composto
# A = "species = virginica"
# B = "sepal_length_cm > 6"
A = df["species"] == "virginica"
B = df["sepal_length_cm"] > 6

resumo_eventos = pd.Series({
    "P(A)": A.mean(),
    "P(B)": B.mean(),
    "P(A ∩ B)": (A & B).mean(),
    "P(A ∪ B)": (A | B).mean(),
    "P(A^c)": (~A).mean()
})
resumo_eventos


P(A)        0.333333
P(B)        0.406667
P(A ∩ B)    0.273333
P(A ∪ B)    0.466667
P(A^c)      0.666667
dtype: float64

### **1.1.6 Probabilidade complementar**
A probabilidade complementar mede a chance de o evento não ocorrer. Ela é útil quando é mais fácil calcular a probabilidade do contrário.

**Fórmula:**
$$
P(A^c) = 1 - P(A)
$$

**Significado dos termos:**
- $P(A)$: probabilidade do evento de interesse;
- $P(A^c)$: probabilidade de o evento não ocorrer.


In [7]:
# exemplo: probabilidade de NÃO selecionar uma observação da espécie versicolor
A = df["species"] == "versicolor"

p_A = A.mean()
p_complementar = 1 - p_A

print(f"P(versicolor): {p_A:.3f}")
print(f"P(não versicolor): {p_complementar:.3f}")
print(f"Verificação direta: {(~A).mean():.3f}")


P(versicolor): 0.333
P(não versicolor): 0.667
Verificação direta: 0.667


### **1.1.7 Regra do produto**
A regra do produto é usada para calcular a probabilidade de interseção. Quando os eventos são dependentes, devemos usar a probabilidade condicional.

**Fórmula geral:**
$$
P(A \cap B) = P(A)P(B \mid A)
$$

**Caso particular de independência:**
$$
P(A \cap B) = P(A)P(B)
$$

**Significado dos termos:**
- $P(B \mid A)$: probabilidade de $B$ ocorrer dado que $A$ já ocorreu.


In [8]:
# exemplo: A = "species = virginica" e B = "petal_length_cm > 5"
A = df["species"] == "virginica"
B = df["petal_length_cm"] > 5

p_A = A.mean()
p_B_dado_A = B[A].mean()
p_intersecao = (A & B).mean()
p_regra_produto = p_A * p_B_dado_A

pd.Series({
    "P(A)": p_A,
    "P(B|A)": p_B_dado_A,
    "P(A ∩ B) observado": p_intersecao,
    "P(A)P(B|A)": p_regra_produto
})


P(A)                  0.333333
P(B|A)                0.820000
P(A ∩ B) observado    0.273333
P(A)P(B|A)            0.273333
dtype: float64

### **1.1.8 Regra da soma**
A regra da soma calcula a probabilidade de pelo menos um entre dois eventos ocorrer. Para evitar dupla contagem, a interseção deve ser subtraída.

**Fórmula:**
$$
P(A \cup B) = P(A) + P(B) - P(A \cap B)
$$

**Significado dos termos:**
- $P(A \cup B)$: probabilidade da união;
- $P(A \cap B)$: parte comum contada duas vezes em $P(A)+P(B)$.


In [9]:
# exemplo: A = "species = setosa" e B = "sepal_width_cm > 3.5"
A = df["species"] == "setosa"
B = df["sepal_width_cm"] > 3.5

p_uniao_observada = (A | B).mean()
p_regra_soma = A.mean() + B.mean() - (A & B).mean()

pd.Series({
    "P(A ∪ B) observado": p_uniao_observada,
    "P(A)+P(B)-P(A∩B)": p_regra_soma
})


P(A ∪ B) observado    0.353333
P(A)+P(B)-P(A∩B)      0.353333
dtype: float64

### **1.1.9 Probabilidade condicional**
Probabilidade condicional mede a chance de um evento ocorrer sob a informação de que outro evento já ocorreu. É um conceito central em modelagem estatística, classificação, risco e inferência.

**Fórmula:**
$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0
$$

**Significado dos termos:**
- $P(A \mid B)$: probabilidade de $A$ dado $B$;
- $P(A \cap B)$: probabilidade conjunta de $A$ e $B$;
- $P(B)$: probabilidade do evento condicionante.


In [10]:
# exemplo: probabilidade de a flor ser virginica dado que a pétala tem mais de 5 cm
A = df["species"] == "virginica"
B = df["petal_length_cm"] > 5

p_A_dado_B = (A & B).mean() / B.mean()

print(f"P(virginica | petal_length_cm > 5): {p_A_dado_B:.3f}")


P(virginica | petal_length_cm > 5): 0.976


### **1.1.10 Teorema de Bayes**
O teorema de Bayes permite inverter condicionais. Em vez de calcular apenas a probabilidade de observar uma evidência dado um grupo, podemos estimar a probabilidade do grupo dado a evidência observada.

**Fórmula:**
$$
P(A \mid B) = \frac{P(B \mid A)P(A)}{P(B)}, \quad P(B) > 0
$$

**Significado dos termos:**
- $P(A)$: probabilidade a priori de $A$;
- $P(B \mid A)$: verossimilhança da evidência $B$ sob $A$;
- $P(B)$: probabilidade total da evidência;
- $P(A \mid B)$: probabilidade a posteriori de $A$ dado $B$.

Esse teorema é a base conceitual de classificadores bayesianos e de muitas rotinas de atualização de crença em estatística e aprendizado de máquina.


In [11]:
# exemplo: teorema de Bayes para P(virginica | petal_length_cm > 5)
A = df["species"] == "virginica"
B = df["petal_length_cm"] > 5

p_A = A.mean()
p_B = B.mean()
p_B_dado_A = B[A].mean()
p_bayes = (p_B_dado_A * p_A) / p_B

pd.Series({
    "P(A)": p_A,
    "P(B)": p_B,
    "P(B|A)": p_B_dado_A,
    "P(A|B) por Bayes": p_bayes
})


P(A)                0.333333
P(B)                0.280000
P(B|A)              0.820000
P(A|B) por Bayes    0.976190
dtype: float64

### **1.1.11 Aplicação do teorema de Bayes**
Considere o seguinte problema: uma observação foi selecionada e sabemos que o comprimento da pétala é maior que 5 cm. Queremos estimar a probabilidade de essa observação pertencer à espécie *virginica*.

Esse tipo de raciocínio aparece em aplicações como:
- diagnóstico médico, ao estimar a doença dado um teste positivo;
- detecção de fraude, ao estimar fraude dado um padrão suspeito;
- classificação automática, ao estimar a classe dado um conjunto de atributos.

No iris, usamos o comprimento da pétala como evidência e a espécie como hipótese. A interpretação final é importante: o resultado não mede certeza absoluta, mas uma probabilidade condicionada à evidência escolhida.


In [12]:
# aplicação: tabela de contingência e interpretação bayesiana
contingencia = pd.crosstab(
    df["species"] == "virginica",
    df["petal_length_cm"] > 5,
    margins=True
)

contingencia.index = ["não virginica", "virginica", "Total"]
contingencia.columns = ["petal<=5", "petal>5", "Total"]
contingencia


,petal<=5,petal>5,Total
não virginica,99,1,100
virginica,9,41,50
Total,108,42,150


---

## **2 Amostragem**
Amostragem é o processo de selecionar uma parte da população para análise. Em ciência de dados, muitas vezes não é viável observar toda a população, seja por custo, tempo, limitação operacional ou indisponibilidade de dados.

Uma boa amostra deve preservar, tanto quanto possível, as características relevantes da população. Quando isso não ocorre, o processo pode introduzir distorções e comprometer inferências, estimativas e modelos.


### **2.1 Principais métodos**
Os principais métodos de amostragem podem ser organizados em probabilísticos e não probabilísticos.

**Amostragem aleatória simples:** cada elemento da população tem a mesma chance de ser selecionado.

**Amostragem estratificada:** a população é dividida em estratos e a seleção ocorre dentro de cada grupo.

**Amostragem sistemática:** seleciona-se um ponto inicial e depois elementos em intervalos regulares.

**Amostragem por conveniência:** seleciona-se o que está mais disponível, sem garantia de representatividade.

Do ponto de vista matemático, se uma amostra aleatória simples de tamanho $n$ é retirada de uma população de tamanho $N$, cada subconjunto de tamanho $n$ tem a mesma probabilidade:
$$
P(	ext{amostra específica}) = \frac{1}{\binom{N}{n}}
$$

**Significado dos termos:**
- $N$: tamanho da população;
- $n$: tamanho da amostra;
- $\binom{N}{n}$: número de amostras possíveis de tamanho $n$.


In [13]:
# exemplo: comparando amostragem aleatória simples e estratificada por espécie
amostra_aleatoria = df.sample(n=15, random_state=42)

amostra_estratificada = (
    df.groupby("species", group_keys=False)
      .apply(lambda x: x.sample(n=5, random_state=42))
      .reset_index(drop=True)
)

print("Distribuição na população:")
print(df["species"].value_counts(normalize=True).sort_index())
print("\nDistribuição na amostra aleatória simples:")
print(amostra_aleatoria["species"].value_counts(normalize=True).sort_index())
print("\nDistribuição na amostra estratificada:")
print(amostra_estratificada["species"].value_counts(normalize=True).sort_index())


Distribuição na população:
species
setosa        0.333333
versicolor    0.333333
virginica     0.333333
Name: proportion, dtype: float64

Distribuição na amostra aleatória simples:
species
setosa        0.4
versicolor    0.4
virginica     0.2
Name: proportion, dtype: float64

Distribuição na amostra estratificada:
species
setosa        0.333333
versicolor    0.333333
virginica     0.333333
Name: proportion, dtype: float64


/var/folders/6p/18wb0ybd0550z_49dfmkhkbw0000gn/T/ipykernel_45686/2011397355.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=5, random_state=42))


### **2.2 Amostragem na prática**
Na prática, amostragem envolve decisões sobre custo, objetivo analítico, tamanho de amostra, variabilidade dos dados e risco de viés.

Uma forma simples de avaliar uma estratégia é comparar estatísticas da amostra com estatísticas da população. Se a amostra for muito pequena ou mal construída, medidas como média e desvio padrão podem se afastar bastante dos valores populacionais.

**Erro amostral para a média:**
$$
	ext{erro} = \bar{x}_{	ext{amostra}} - \mu_{	ext{população}}
$$

**Significado dos termos:**
- $\bar{x}_{	ext{amostra}}$: média estimada a partir da amostra;
- $\mu_{	ext{população}}$: média real da população.


In [14]:
# exemplo: efeito do tamanho da amostra na estimativa da média
col = "petal_length_cm"
media_populacao = df[col].mean()

resultados = []
for tamanho in [5, 10, 20, 30, 50]:
    media_amostra = df.sample(n=tamanho, random_state=42)[col].mean()
    erro = media_amostra - media_populacao
    resultados.append([tamanho, media_amostra, media_populacao, erro])

pd.DataFrame(resultados, columns=["n_amostra", "media_amostra", "media_populacao", "erro"])


,n_amostra,media_amostra,media_populacao,erro
0,5,4.520000,3.758,0.762000
1,10,4.120000,3.758,0.362000
2,20,3.825000,3.758,0.067000
3,30,3.883333,3.758,0.125333
4,50,3.642000,3.758,-0.116000


### **2.3 Representatividade da amostra**
Uma amostra é representativa quando preserva propriedades relevantes da população para o objetivo do estudo. Representatividade não significa cópia perfeita, mas semelhança suficiente para que inferências sejam razoáveis.

Em problemas de classificação, por exemplo, é importante que grupos relevantes não desapareçam da amostra. Em termos práticos, comparar proporções por categoria é um critério inicial útil.

**Diferença de proporções:**
$$
\Delta = \hat{p}_{	ext{amostra}} - p_{	ext{população}}
$$

**Significado dos termos:**
- $\hat{p}_{	ext{amostra}}$: proporção observada na amostra;
- $p_{	ext{população}}$: proporção real da população;
- $\Delta$: desvio entre amostra e população.


In [15]:
# exemplo: avaliando representatividade por espécie
prop_pop = df["species"].value_counts(normalize=True).sort_index()
prop_amostra = amostra_aleatoria["species"].value_counts(normalize=True).sort_index()

comparacao = pd.DataFrame({
    "proporcao_populacao": prop_pop,
    "proporcao_amostra": prop_amostra
}).fillna(0)

comparacao["diferenca"] = comparacao["proporcao_amostra"] - comparacao["proporcao_populacao"]
comparacao


,proporcao_populacao,proporcao_amostra,diferenca
species,,,
setosa,0.333333,0.4,0.066667
versicolor,0.333333,0.4,0.066667
virginica,0.333333,0.2,-0.133333


### **2.4 Viés de amostragem**
Viés de amostragem ocorre quando o processo de seleção favorece sistematicamente certos elementos da população. Nesses casos, a amostra deixa de representar adequadamente o conjunto de interesse.

Exemplos comuns incluem:
- coletar apenas indivíduos mais fáceis de acessar;
- selecionar dados de um único período, ignorando sazonalidade;
- construir conjuntos de treino e teste com distribuições muito diferentes.

O problema central é que o erro deixa de ser apenas aleatório e passa a ser sistemático.

**Representação conceitual do viés:**
$$
	ext{viés} = E(\hat{	heta}) - 	heta
$$

**Significado dos termos:**
- $\hat{	heta}$: estimador obtido pela amostra;
- $E(\hat{	heta})$: valor esperado do estimador;
- $	heta$: parâmetro verdadeiro da população.


In [16]:
# exemplo: criando uma amostra enviesada ao selecionar apenas pétalas longas
amostra_enviesada = df[df["petal_length_cm"] > 4.8]

comparacao_vies = pd.DataFrame({
    "populacao": df["species"].value_counts(normalize=True).sort_index(),
    "amostra_enviesada": amostra_enviesada["species"].value_counts(normalize=True).sort_index()
}).fillna(0)

comparacao_vies["diferenca"] = comparacao_vies["amostra_enviesada"] - comparacao_vies["populacao"]
comparacao_vies


,populacao,amostra_enviesada,diferenca
species,,,
setosa,0.333333,0.000000,-0.333333
versicolor,0.333333,0.078431,-0.254902
virginica,0.333333,0.921569,0.588235


---

## **Encerramento**
Nesta aula, você estudou conceitos centrais de probabilidade e amostragem, incluindo eventos, independência, probabilidade condicional, teorema de Bayes, métodos de amostragem, representatividade e viés. Esses tópicos formam a base para inferência estatística, avaliação de modelos e tomada de decisão em ciência de dados.


---

## **Exercícios propostos**

**Exercício 1.** Considere o evento $A$ = "a observação pertence à espécie *versicolor*" e o evento $ B = \{\texttt{sepal\_width\_cm} > 3{,}0\} $. Calcule $P(A)$, $P(B)$, $P(A \cap B)$ e $P(A \mid B)$. Em seguida, interprete se a informação de $B$ altera de forma relevante a chance de ocorrência de $A$.

**Exercício 2.** Extraia uma amostra aleatória simples com 20 observações e compare a média de `petal_width_cm` da amostra com a média populacional. Repita o procedimento para uma amostra com 60 observações e compare os erros amostrais.

**Exercício 3.** Construa uma amostra estratificada com 6 observações por espécie. Depois, compare a distribuição das espécies na população, na amostra estratificada e em uma amostra por conveniência formada pelas primeiras 18 linhas do dataset. Discuta qual delas é mais representativa e por quê.


In [17]:
# apoio opcional: tabela-resumo para consulta durante os exercícios
resumo_final = {
    "n_populacao": len(df),
    "media_petal_length": df["petal_length_cm"].mean(),
    "media_petal_width": df["petal_width_cm"].mean(),
    "proporcoes_species": df["species"].value_counts(normalize=True).to_dict()
}

resumo_final


{'n_populacao': 150,
 'media_petal_length': np.float64(3.7580000000000005),
 'media_petal_width': np.float64(1.1993333333333336),
 'proporcoes_species': {np.str_('setosa'): 0.3333333333333333,
  np.str_('versicolor'): 0.3333333333333333,
  np.str_('virginica'): 0.3333333333333333}}

---

# **Referências**

* NEMEC, Fernando. Estatística para Data Science e Inteligência Artificial: os fundamentos estatísticos que você precisa para trabalhar com dados. São Paulo: Nemec Consultoria, 2025.

* DEVORE, Jay L. Probabilidade e estatística para engenharia e ciências. 9. ed. São Paulo: Cengage Learning, 2018.

* MONTGOMERY, Douglas C.; RUNGER, George C. Estatística aplicada e probabilidade para engenheiros. 7. ed. Rio de Janeiro: LTC, 2018.

* JAMES, Gareth; WITTEN, Daniela; HASTIE, Trevor; TIBSHIRANI, Robert. An introduction to statistical learning: with applications in Python. Cham: Springer, 2023.


---

© 2026 Jones Egydio. Todos os direitos reservados.  

Conecte-se comigo no [LinkedIn](https://www.linkedin.com/in/jones-egydio-msc-3300359/).
